# 2Dify AI — Private Colab

Upload a video, convert the full scene to hand-drawn 2D with the Lucy V2V model, remove the original audio, and download the silent MP4. Google Drive stores checkpoints so an interrupted run can resume.

**Important:** this uses a remote GPU model. Free GPU availability/quotas are not guaranteed.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/2Dify')
ROOT.mkdir(parents=True, exist_ok=True)
print('Drive workspace:', ROOT)

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install -U gradio_client requests

In [ ]:
REPO = Path('/content/2dify-repo')
if not REPO.exists():
    !git clone -q https://github.com/mukz3467/Multi-tool-Ai-.git /content/2dify-repo
else:
    !git -C /content/2dify-repo pull -q
WORKER = REPO / '2dify' / '2dify_worker.py'
print('Worker:', WORKER)

## 1) Upload or select a video

For a new job choose **U** and upload the video. If Colab restarts, reconnect Drive and select the same video; the worker will reuse completed chunks.

In [ ]:
videos = sorted([p for p in ROOT.iterdir() if p.is_file() and p.suffix.lower() in {'.mp4','.mov','.mkv','.webm','.avi'}])
if videos:
    print('Videos already in Drive:')
    for i, p in enumerate(videos):
        print(f'{i}: {p.name}')
else:
    print('No videos found in Drive yet.')

choice = input('Choose video number, or type U to upload: ').strip()
if choice.lower() == 'u':
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No video uploaded')
    name = next(iter(uploaded))
    INPUT = ROOT / name
    shutil.copy2(name, INPUT)
elif choice.isdigit() and int(choice) < len(videos):
    INPUT = videos[int(choice)]
else:
    raise ValueError('Invalid selection')

OUTPUT = ROOT / (INPUT.stem + '_2dify_silent.mp4')
print('Input:', INPUT)
print('Output:', OUTPUT)
print('Checkpoints:', OUTPUT.parent / (OUTPUT.stem + '.2dify_chunks'))

## 2) Convert

Run the next cell. Each model window is about 3.4 seconds. The worker saves each completed window to Google Drive and skips completed windows on resume.

In [ ]:
if not WORKER.exists():
    raise FileNotFoundError(f'Worker not found: {WORKER}')
!python "$WORKER" "$INPUT" "$OUTPUT"

In [ ]:
if OUTPUT.exists() and OUTPUT.stat().st_size > 1024:
    print('Done:', OUTPUT)
    print('Size (MB):', round(OUTPUT.stat().st_size / 1024 / 1024, 2))
    files.download(str(OUTPUT))
else:
    print('Final output is not complete yet. Keep the checkpoint folder and resume the same job later.')